In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [2]:
gdf = gpd.read_file('../../dataset/shp/final_version.shp')
gdf

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,distance,dist_km,dist_norm,...,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population,area_km2,density_km,geometry
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,23646.846826,23.646847,0.533159,...,2071.724079,0.997450,0.451028,0.681208,0.321536,1.0,1159,0.026,45062.21,"POLYGON ((-5162009.388 -2692597.734, -5162042...."
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,2544.012189,2.544012,0.080354,...,1167.193154,1.000000,0.978808,0.872972,0.999800,0.0,340,0.117,2916.00,"POLYGON ((-5460228.537 -3115389.701, -5460231...."
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,28454.700968,28.454701,0.507651,...,1003.263292,1.000000,1.000000,0.853671,0.999800,0.0,394,0.028,14189.51,"POLYGON ((-4840464.531 -2618564.009, -4840451...."
3,33045570104,Parque Nossa Senhora da Penha,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,1286.545036,1.286545,0.018809,...,659.135306,0.990750,0.892921,0.579645,1.000000,0.0,1058,0.016,65726.53,"POLYGON ((-4810854.603 -2616904.986, -4810851...."
4,35095020135,Núcleo Residencial Jardim das Andorinhas II,35,São Paulo,SP,3509502,Campinas,5245.591377,5.245591,0.146824,...,1488.467146,0.991000,0.699696,0.828075,0.999800,1.0,658,0.037,17788.59,"POLYGON ((-5233560.28 -2622317.23, -5233639.32..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12339,27043020123,Vila Santa Cruz,27,Alagoas,AL,2704302,Maceió,12464.347471,12.464347,0.490099,...,2773.897971,0.854507,0.545140,0.804361,0.999800,1.0,585,0.030,19327.98,"POLYGON ((-3981406.076 -1070408.33, -3981639.1..."
12340,28003080047,Invasão da Jabotiana,28,Sergipe,SE,2800308,Aracaju,5943.694057,5.943694,0.187358,...,2430.944342,0.974801,0.385397,0.844824,0.999800,1.0,334,0.019,17261.87,"POLYGON ((-4128645.906 -1225338.301, -4128644...."
12341,35188000223,Residencial Bambi II,35,São Paulo,SP,3518800,Guarulhos,18639.788181,18.639788,0.949105,...,5287.866697,0.704965,0.532710,0.804146,0.999800,2.0,390,0.101,3868.86,"POLYGON ((-5164614.754 -2677936.544, -5164656...."
12342,26116060881,Sítio Wanderley,26,Pernambuco,PE,2611606,Recife,7352.563603,7.352564,0.570652,...,811.613421,0.930403,0.704766,0.848365,0.997201,1.0,2015,0.104,19323.35,"POLYGON ((-3890716.958 -898474.936, -3890713.9..."


In [3]:
def gini_lorenz(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    # Gini is defined for non-negative values
    x = np.clip(x, 0, None)

    n = x.size
    if n == 0:
        return np.nan
    s = x.sum()
    if s == 0:
        return 0.0  # all zeros -> perfect equality

    x_sorted = np.sort(x)

    # Lorenz curve points
    Y = np.concatenate([[0.0], np.cumsum(x_sorted) / s])  # length n+1
    X = np.linspace(0.0, 1.0, n + 1)                      # 0, 1/n, ..., 1

    # Trapezoidal integration form:
    # G = 1 - sum_{k=0}^{n-1} (X_{k+1}-X_k) * (Y_{k+1}+Y_k)
    dX = np.diff(X)
    G = 1.0 - np.sum(dX * (Y[1:] + Y[:-1]))
    return float(G)

In [4]:
cluster_col = "cluster_3"
cols = {
    "water_inequality": "wtr_idx",
    "sanitation_inequality": "sanit_idx",
    "living_area_inequality": "living_area_idx",
    "housing_durability_inequality": "durable_house_idx",
}

gini_by_cluster = (
    exp_df_inv
    .groupby(cluster_col)
    .apply(lambda df: pd.Series({k: gini_lorenz(df[v].values) for k, v in cols.items()}))
    .reset_index()
)

gini_by_cluster

NameError: name 'exp_df_inv' is not defined

In [5]:
def gini_lorenz_brown(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]

    # Gini (Lorenz) assumes non-negative values
    if np.any(x < 0):
        raise ValueError("Gini via Lorenz requires non-negative values. Found x < 0.")

    n = x.size
    if n == 0:
        return np.nan

    total = x.sum()
    if total == 0:
        return 0.0  # perfect equality (all zeros)

    x_sorted = np.sort(x)  # x_(i)

    # Lorenz points
    # X_k = k/n for k=0..n
    X = np.arange(n + 1) / n
    # Y_k = cumulative share of the attribute (index)
    Y = np.empty(n + 1, dtype=float)
    Y[0] = 0.0
    Y[1:] = np.cumsum(x_sorted) / total

    # Brown trapezoid form
    dX = X[1:] - X[:-1]        # = 1/n
    G = 1.0 - np.sum(dX * (Y[1:] + Y[:-1]))
    return float(G)


# --- Apply to your GeoDataFrame/DataFrame ---
# Expected columns: wtr_idx, sanit_idx, liv_a_idx, hh_idx, cluster, population
# (population is NOT used in the unweighted Lorenz Gini)

def gini_by_cluster(df: pd.DataFrame,
                    cluster_col: str = "cluster",
                    index_cols = ("wtr_idx", "sanit_idx", "liv_a_idx", "hh_idx")) -> pd.DataFrame:
    rows = []
    for cl, sub in df.groupby(cluster_col, dropna=False):
        row = {"cluster": cl, "n_settlements": len(sub)}
        for col in index_cols:
            row[f"gini_{col}"] = gini_lorenz_brown(sub[col].to_numpy())
        rows.append(row)
    out = pd.DataFrame(rows).sort_values("cluster").reset_index(drop=True)
    return out


# Example usage:
gini_table = gini_by_cluster(
    gdf,
    cluster_col="cluster",
    index_cols=("wtr_idx", "sanit_idx", "liv_a_idx", "hh_idx")
)
print(
    gini_table.to_string(
        index=False,
        float_format="%.6f".__mod__  # 6 casas decimais, fácil de mudar
    )
)

 cluster  n_settlements  gini_wtr_idx  gini_sanit_idx  gini_liv_a_idx  gini_hh_idx
0.000000           6139      0.018172        0.043577        0.055614     0.011967
1.000000           4576      0.038366        0.100368        0.047059     0.010836
2.000000           1629      0.154343        0.105914        0.052410     0.013711


In [18]:
sub[col].to_numpy()

array([0.99980004, 0.99980004, 0.98970206, ..., 0.99980004, 0.99980004,
       0.99980004], shape=(1629,))

In [8]:
def gini_lorenz_weighted(x, w):
    """
    Population-weighted Gini coefficient using the Lorenz curve (Brown formulation).

    Parameters
    ----------
    x : array-like
        Non-negative index values (e.g., wtr_idx) per settlement.
    w : array-like
        Population weights per settlement.

    Returns
    -------
    float
        Population-weighted Gini coefficient.
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    # Remove invalid entries
    mask = (~np.isnan(x)) & (~np.isnan(w)) & (w > 0)
    x, w = x[mask], w[mask]

    if x.size == 0:
        return np.nan

    if np.any(x < 0):
        raise ValueError("Index values must be non-negative.")

    # Order by index value
    order = np.argsort(x)
    x_ord = x[order]
    w_ord = w[order]

    # Totals
    W = w_ord.sum()
    WX = np.sum(w_ord * x_ord)

    if WX == 0:
        return 0.0  # perfect equality

    # Lorenz curve coordinates
    X = np.concatenate([[0.0], np.cumsum(w_ord) / W])
    Y = np.concatenate([[0.0], np.cumsum(w_ord * x_ord) / WX])

    # Brown trapezoidal formula
    dX = X[1:] - X[:-1]
    G = 1.0 - np.sum(dX * (Y[1:] + Y[:-1]))

    return float(G)

In [9]:
index_cols = {
    "water_inequality": "wtr_idx",
    "sanitation_inequality": "sanit_idx",
    "living_area_inequality": "liv_a_idx",
    "housing_durability_inequality": "hh_idx",
}

results = []

for cl, sub in gdf.groupby("cluster"):
    row = {
        "cluster": cl,
        "n_settlements": len(sub),
        "population": sub["population"].sum()
    }

    for name, col in index_cols.items():
        row[name] = gini_lorenz_weighted(
            sub[col].values,
            sub["population"].values
        )

    results.append(row)

gini_population_weighted = pd.DataFrame(results).sort_values("cluster")
gini_population_weighted

,cluster,n_settlements,population,water_inequality,sanitation_inequality,living_area_inequality,housing_durability_inequality
0,0.0,6139,8958140,0.016683,0.045670,0.060928,0.009728
1,1.0,4576,5453919,0.044106,0.090047,0.051622,0.009505
2,2.0,1629,1977246,0.120347,0.089126,0.052389,0.009518
